## Cruise Control Problem

This notebook is a **public companion** for the **cruise-control** benchmark in:

*Meta-Reinforcement Learning for Robust and Non-greedy Control Barrier Functions in Spacecraft Proximity Operations* (TAES submission)

It covers:
- Problem definition and parameters (as used in the paper)
- Loading a **fixed Monte Carlo episode bank**
- Evaluating three controllers on the same episode set:
  1. **Untuned ICCBF**
  2. **MLP-tuned ICCBF**
  3. **RNN (LSTM) tuned ICCBF**
- Producing paper-style metrics and plots (thrust distributions, success rates, etc.)

Expected inputs (edit paths in `CONFIG`):
- `data/cruise_control_episode_bank.npz`

###  Problem Definition 

![Cruise Control Problem](images/cruisecontrol.drawio.png)


This problem considers a point-mass model of a vehicle moving along a straight line. A following vehicle trails a lead vehicle at a distance $d$, with the lead vehicle travelling at a known constant speed $v_0$. The objective is to design a controller that drives the following vehicle to the speed limit $v_{max}$ while ensuring collision avoidance. The collision avoidance safety constraint is specified as $d \geq 1.8v$, and the CLF constraint that drives the following vehicle to the speed limit is defined as $V(x)  =(v - v_{\text{max}})^2$. {No drag is considered in this preliminary cruise control scenario.} Defining the state vector as $x = [d,\, v]^T$, the dynamical model is

\begin{equation}
\begin{bmatrix}
\dot{d} \\
\dot{v}
\end{bmatrix}
=
\begin{bmatrix}
v_0 - v \\
-\tfrac{F(v)}{m}
\end{bmatrix}
+
\begin{bmatrix}
0 \\
g_0
\end{bmatrix} u,
\qquad
\mathcal{U} = \{u : |u| \leq u_{max},
\end{equation}
where $u$ is the control input. The resistive force $F(v)$ is modeled as $F(v) = f_0 + f_1 v + f_2 v^2$, $m$ is the vehicle mass, and $g_0$ is the gravitational acceleration.  The safe set $\mathcal{S}$ is then defined as
\begin{equation}
\mathcal{S} = \{x \in \mathcal{X} \;|\; h_0(x) = x_1 - 1.8x_2 \geq 0\}.
\end{equation}



## Path Setup 

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[0]   # notebooks/ -> repo root
SRC = REPO_ROOT / "src"
sys.path.insert(0, str(SRC))  # do this once per kernel

## Initial State Distribution 

The initial states are drawn such that $\mathbf{x}_0 \in \mathcal{S}$. The episode parameter vector
includes the vehicle mass $m$, the front-vehicle (initial) speed $v_0$, the target speed $v_{\max}$, and the maximum
available thrust $u_{\max}$:
\begin{equation}
    \mathbf{p} = [m,\ v_0,\ v_{\max},\ u_{\max}]^\top.
\end{equation}

At the start of each episode, the parameters are sampled from a uniform distribution , with
\begin{equation}
   \mathbf{p}_{\min} = [1320.0,\ 12.501,\ 21.6,\ 0.20]^\top
\end{equation}

\begin{equation}
    \mathbf{p}_{\max} = [1980.0,\ 15.279,\ 26.4,\ 0.30]^\top
\end{equation}

These bounds correspond to a $\pm20\%$ variation in $m$ and $u_{\max}$ and a $\pm10\%$ variation in $v_0$ and $v_{\max}$ relative to the nominal values provided in \url{https://dev10110.github.io/pdfs/2021-iccbfs.pdf}.

To generate the initial state distribution run:

In [ ]:
from cruisecontrol.make_fixed_ics import make_cruise_control_episode_bank
make_cruise_control_episode_bank()

plot to parameter distribution:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

D = np.load("../src/data/cruise_control/cruise_control_episode_bank.npz", allow_pickle=True)
print(D.files)
for k in D.files:
    v = D[k]
    print(k, getattr(v, "shape", None), v.dtype)

def plot_meta_params_from_bank(npz_path, bins=60):
    D = np.load(npz_path, allow_pickle=True)

    if "meta_params" not in D.files:
        raise KeyError(f"'meta_params' not found. Keys are: {D.files}")

    P = np.asarray(D["meta_params"]).astype(float)
    names = D["meta_param_names"].tolist() if "meta_param_names" in D.files else ["m","v0","vmax","umax"]

    titles = {
        "m": r"Mass $m$",
        "v0": r"Front speed $v_0$",
        "vmax": r"Target speed $v_{\max}$",
        "umax": r"Max thrust $u_{\max}$",
    }

    fig, axes = plt.subplots(2, 2, figsize=(10, 6))
    axes = axes.ravel()

    for i in range(4):
        ax = axes[i]
        x = P[:, i]
        key = names[i]
        ax.hist(x, bins=bins)
        ax.set_title(titles.get(key, key))
        mu, sd, med = np.mean(x), np.std(x), np.median(x)
        ax.text(0.02, 0.95, f"μ={mu:.4g}, σ={sd:.4g}, med={med:.4g}, N={x.size}",
                transform=ax.transAxes, va="top", fontsize=9)
        ax.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.show()

plot_meta_params_from_bank("../src/data/cruise_control/cruise_control_episode_bank.npz", bins=60)


### Training

Here we train an MLP network and an LSTM network to determine the class-k function constants used in the ICCBF cascade. Note that the given settings are the ones used to generate the results in the paper. 
 
To run MLP training run the following code.

In [ ]:
from cruisecontrol.train_CCNN import train_cruise_control

model, log_dir = train_cruise_control(
    # =========================
    # Nominal inputs (explicit)
    # =========================
    dt=0.1,
    total_episodes=200_000,
    approx_episode_len=200,
    seed=123,

    # ---- Architecture ----
    layers=4,
    nodes=64,
    activation_fn="tanh",
    policy_type="PPO",

    # ---- RL hyperparameters ----
    learning_rate=1e-4,
    lr_type="C",
    gamma=0.999,
    gae_lambda=0.99,
    clip_range=0.1,
    ent_coef=0.01,
    n_epochs=10,
    batch_size=512,
    n_steps=256,
    target_kl=0.02,

    # ---- Environment ----
    num_env=32,
    trainON=True,
    trainLoad=False,

    # ---- Logging ----
    root_log_dir="TrainedModels",
)

Then to run the training process with the LSTM RNN network, run the following code.

In [ ]:
from cruisecontrol.train_CCRNN import train_cruise_control_lstm

model, log_dir = train_cruise_control_lstm(
    # =========================
    # Nominal inputs (explicit)
    # =========================
    dt=0.1,
    seed=123,

    # ---- Network architecture ----
    layers=4,
    nodes=64,
    activation_fn=activation.Tanh,

    # ---- LSTM parameters ----
    lstm_hidden_size=64,
    n_lstm_layers=1,
    shared_lstm=False,
    enable_critic_lstm=True,

    # ---- RL hyperparameters ----
    total_episodes=100_000,
    approx_episode_len=200,
    learning_rate=1e-4,
    lr_type="C",
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.1,
    ent_coef=0.01,
    target_kl=0.02,
    n_epochs=10,
    batch_size=512,
    n_steps=200,   # must be divisible by num_env

    # ---- Parallelism ----
    num_env=32,

    # ---- Control ----
    trainON=True,
    trainLoad=False,

    # ---- Logging ----
    root_log_dir="TrainedModels",
)


### Evaluation 
Evaluate the trained MLP and LSTM models on the 5000 sample test cases generated 

In [1]:
# MLP
from cruisecontrol.eval_parallel import EvalConfig, evaluate_parallel

cfg = EvalConfig(
    ics_path=str("../src/data/cruise_control/cruise_control_episode_bank.npz"),
    model_path=str( "../src/cruisecontrol/TrainedModels/MetaCNNCruiseControlMargin_PPO_L3_N64_Tanh_lr0.0001_g0.999_gae0.990_ent0.01/best_model.zip"),  # update
    dt=0.1,
    TOF=40.0,
    n_workers=8,
    n_chunks=32,
    out_dir=str("../src/data/cruise_control"),
    out_mat="MLPsol.mat",
)

out_mat_path = evaluate_parallel(cfg)
print(out_mat_path)

ModuleNotFoundError: No module named 'cruisecontrol'

In [ ]:
# LSTM
from cruisecontrol.eval_parallel import EvalConfig, evaluate_parallel

cfg = EvalConfig(
    ics_path=str("../src/data/cruise_control/cruise_control_episode_bank.npz"),
    model_path=str( "../src/cruisecontrol/TrainedModels/MarginMetaRNNwNoiseCruiseControl_CLFhigherweight/best_model.zip"),  # update
    dt=0.1,
    TOF=40.0,
    n_workers=8,
    n_chunks=32,
    out_dir=str(REPO_ROOT / "ResultsEval"),
    out_dir=str("../src/data/cruise_control"),
    out_mat="LSTMsol.mat",
)

out_mat_path = evaluate_parallel(cfg)
print(out_mat_path)

### Results 

This plots the generated results

In [ ]:
path_iccbf   = "../src/data/cruise_control/noRLsol.mat"
path_stage1  = "../src/data/cruise_control/MLPsol.mat"
path_rnn     = "../src/data/cruise_control/LSTMsol.mat"
path_gru     = "../ResultsEval/eval_GRUTunedICCBF_CruiseControl_20260305_125057_parallel_20260310_105730.mat"
path_mamba   = "../ResultsEval/eval_Mamba2TunedICCBF_CruiseControl_20260225_084139_parallel_20260225_225335.mat"

from metarl_iccbf.cruise_control.analysis.plot_cc_results import plot_cruisecontrol

out = plot_cruisecontrol(
    mat_paths=[path_iccbf, path_stage1, path_rnn, path_gru, path_mamba],
    model_names=("ICCBF", "MLP-tuned ICCBF", "RNN-tuned ICCBF", "GRU-tuned ICCBF", "Mamba-tuned ICCBF"),
    save_prefix=None,  # or "figs/cc_level4" to save PNGs
)

print(out["latex_table"])

## DA margin Validation


To validate the accuracy of the DA interval enclosures used in the sampled-data margin computation, a MC analysis is performed along complete trajectories. For each MC episode $e$ and each discrete time step $t=kT$, the DA polynomials are re-expanded locally about the current sampled state $\mathbf{x}_{e,k}$. For the margin calculation, using DA bounding over $\mathcal{B}(\mathbf{x}_{e,k},r)$, conservative interval enclosures $[\ell_{e,k},u_{e,k}]$ are computed for
$q \in \{\, b_N,\; L_f b_N,\; \boldsymbol{L}_g b_N\},$

For each quantity \(q\), the slack at episode  $e$ and time index $k$ is defined as
\begin{equation}
s_{e,k} \;=\; \min\!\big(q_{e,k}-\ell_{e,k},\;u_{e,k}-q_{e,k}\big),
\end{equation}
where $s_{e,k}<0$ indicates a violation of the DA enclosure. To capture worst-case behavior across the MC set at each time step, the minimum slack over episodes is computed as
\begin{equation}
s_k^{\min} \;=\; \min_{e}\, s_{e,k}.
\end{equation}
The validation results are summarized using violin plots of \(\{s_k^{\min}\}_{k=1}^{K}\) for each quantity \(q\). A strictly positive violin plot indicates that the DA interval bounds remain valid across all MC trajectories and time steps under the chosen $\mathcal{B}(\mathbf{x}_{e,k},r)$, providing empirical support that the DA-derived enclosures used to construct $\widehat{l}_{(\cdot)}(\mathbf{x}_k), \widehat{\Delta} \mathbf{x}_k$, and $\widehat{\nu}(T,\mathbf{x}_k)$ are conservative along the evaluated trajectories.


<p align="center">
  <img src="images/CCDAapprox.png" alt="Distribution of distance to the computed upper and lower DA bounds for each quantity used to calculate the margin $\nu$, over all the MC episodes and all the time steps." width="55%"/>
</p>


